**IN:** directory of pdfs we want to chunk and vectorize

**OUT:** chromadb vector databases for each paper, named by their ID

format:
- chromadb
  - [collection "BO1005"]
  - [collection "JO1013"]
  - ...



In [1]:
# Import libraries
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
import os
from chromadb.utils import embedding_functions

/Users/pk_3/miniconda3/envs/alb/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

CHROMA_PATH = "/Users/pk_3/My_Documents/CTC/Alliance-Bioversity-CIAT/chroma_db"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = chroma_client.get_or_create_collection(
    name="papers",
    embedding_function=embedding_fn
)

pdf_files = [f for f in os.listdir("pdfs") if f.endswith('.pdf')]

for pdf_file in pdf_files:
    PAPER_ID = pdf_file.split('-')[0]
    PDF_PATH = f"pdfs/{pdf_file}"

    loader = PyPDFLoader(PDF_PATH)
    raw_documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100,
    )
    chunks = text_splitter.split_documents(raw_documents)

    documents = []
    ids = []
    metadatas = []

    for i, chunk in enumerate(chunks):
        documents.append(chunk.page_content)
        ids.append(f"{PAPER_ID}_chunk_{i}")
        metadatas.append({"paper_id": PAPER_ID})   # FIXED: metadata added

    collection.upsert(
        documents=documents,
        ids=ids,
        metadatas=metadatas
    )